# MV Validation: Label Split Leakage / Stability Check

This notebook section is designed to be appended to the end of `lgbm_model.ipynb`.

Assumption: the original `lgbm_model.ipynb` has already been run, so `features_df` is already available in the notebook session.

Purpose:
- Check whether the same employee-account pair crosses train / validation / test splits.
- Check whether overlapping windows for the same employee-account pair cross splits.
- Propose a deterministic employee-account-level split.
- Compare original MD split balance vs proposed MV split balance.


## 1. Prepare split check data

In [ ]:
# COMMAND ----------
# MV Validation: Label Split Leakage / Stability Check

import pyspark.sql.functions as F

split_check_df = (
    features_df
    .select(
        "login_id",
        "acct_nbr",
        "date",
        "lookback_window_start",
        "fraud_date",
        "label",
        "label_split",
        "cv_fold"
    )
    .withColumn("date", F.to_date("date"))
    .withColumn("lookback_window_start", F.to_date("lookback_window_start"))
    .withColumn("fraud_date", F.to_date("fraud_date"))
    .withColumn("window_start", F.col("lookback_window_start"))
    .withColumn("window_end", F.coalesce(F.col("fraud_date"), F.col("date")))
)

display(split_check_df.limit(10))

## 2. Current MD split label balance

In [ ]:
# COMMAND ----------
# Current MD split label balance

current_split_balance = (
    split_check_df
    .groupBy("label_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        (F.count("*") - F.sum(F.col("label"))).alias("num_negative"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("label_split")
)

display(current_split_balance)

## 3. Check whether the same employee-account pair appears in multiple splits

In [ ]:
# COMMAND ----------
# Check whether the same employee-account pair appears in multiple splits

pair_split_check = (
    split_check_df
    .groupBy("login_id", "acct_nbr")
    .agg(
        F.count("*").alias("num_rows"),
        F.countDistinct("label_split").alias("num_splits"),
        F.collect_set("label_split").alias("splits"),
        F.sum(F.col("label")).alias("num_positive"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

pair_cross_split = (
    pair_split_check
    .filter(F.col("num_splits") > 1)
    .orderBy(F.desc("num_splits"), F.desc("num_rows"))
)

display(pair_cross_split)

pair_cross_split_summary = (
    pair_split_check
    .agg(
        F.count("*").alias("num_employee_account_pairs"),
        F.sum(F.when(F.col("num_splits") > 1, 1).otherwise(0)).alias("num_pairs_crossing_splits"),
        F.mean(F.when(F.col("num_splits") > 1, 1).otherwise(0)).alias("pct_pairs_crossing_splits")
    )
)

display(pair_cross_split_summary)

## 4. Check whether overlapping windows for the same employee-account pair cross splits

In [ ]:
# COMMAND ----------
# Check whether overlapping windows for the same employee-account pair cross splits

window_base = (
    split_check_df
    .select(
        "login_id",
        "acct_nbr",
        "date",
        "window_start",
        "window_end",
        "label",
        "label_split"
    )
    .dropDuplicates()
    .withColumn("row_id", F.monotonically_increasing_id())
)

a = window_base.alias("a")
b = window_base.alias("b")

overlap_cross_split = (
    a.join(
        b,
        on=[
            F.col("a.login_id") == F.col("b.login_id"),
            F.col("a.acct_nbr") == F.col("b.acct_nbr"),
            F.col("a.row_id") < F.col("b.row_id"),
            F.col("a.label_split") != F.col("b.label_split"),
            F.col("a.window_start") <= F.col("b.window_end"),
            F.col("b.window_start") <= F.col("a.window_end")
        ],
        how="inner"
    )
    .select(
        F.col("a.login_id").alias("login_id"),
        F.col("a.acct_nbr").alias("acct_nbr"),

        F.col("a.date").alias("date_a"),
        F.col("a.window_start").alias("window_start_a"),
        F.col("a.window_end").alias("window_end_a"),
        F.col("a.label_split").alias("label_split_a"),
        F.col("a.label").alias("label_a"),

        F.col("b.date").alias("date_b"),
        F.col("b.window_start").alias("window_start_b"),
        F.col("b.window_end").alias("window_end_b"),
        F.col("b.label_split").alias("label_split_b"),
        F.col("b.label").alias("label_b")
    )
)

display(overlap_cross_split.limit(100))

overlap_cross_split_summary = (
    overlap_cross_split
    .agg(
        F.count("*").alias("num_overlapping_window_pairs_crossing_splits"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs_with_overlap_cross_split")
    )
)

display(overlap_cross_split_summary)

## 5. Proposed deterministic employee-account-level split

In [ ]:
# COMMAND ----------
# Proposed deterministic employee-account-level split

proposed_split_df = (
    split_check_df
    .withColumn(
        "pair_hash_bucket",
        F.pmod(
            F.abs(
                F.xxhash64(
                    F.col("login_id").cast("string"),
                    F.col("acct_nbr").cast("string")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "mv_pair_level_split",
        F.when(F.col("pair_hash_bucket") < 70, F.lit("train"))
         .when(F.col("pair_hash_bucket") < 85, F.lit("val"))
         .otherwise(F.lit("test"))
    )
)

proposed_split_balance = (
    proposed_split_df
    .groupBy("mv_pair_level_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        (F.count("*") - F.sum(F.col("label"))).alias("num_negative"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("mv_pair_level_split")
)

display(proposed_split_balance)

## 6. Confirm no employee-account pair crosses the proposed split

In [ ]:
# COMMAND ----------
# Confirm no employee-account pair crosses the proposed split

proposed_pair_check = (
    proposed_split_df
    .groupBy("login_id", "acct_nbr")
    .agg(
        F.countDistinct("mv_pair_level_split").alias("num_proposed_splits"),
        F.collect_set("mv_pair_level_split").alias("proposed_splits")
    )
)

proposed_pair_leakage = (
    proposed_pair_check
    .filter(F.col("num_proposed_splits") > 1)
)

display(proposed_pair_leakage)

proposed_pair_leakage_summary = (
    proposed_pair_check
    .agg(
        F.count("*").alias("num_employee_account_pairs"),
        F.sum(
            F.when(F.col("num_proposed_splits") > 1, 1).otherwise(0)
        ).alias("num_pairs_crossing_proposed_splits")
    )
)

display(proposed_pair_leakage_summary)

## 7. Compare original MD split vs proposed MV pair-level split

In [ ]:
# COMMAND ----------
# Compare original MD split vs proposed MV pair-level split

print("Original MD split balance")
display(current_split_balance)

print("Proposed MV pair-level split balance")
display(proposed_split_balance)

## 8. Create full feature dataframe with proposed MV pair-level split

In [ ]:
# COMMAND ----------
# Create full feature dataframe with proposed MV pair-level split

features_with_mv_split = (
    features_df
    .join(
        proposed_split_df
        .select(
            "login_id",
            "acct_nbr",
            "date",
            "mv_pair_level_split"
        )
        .dropDuplicates(),
        on=["login_id", "acct_nbr", "date"],
        how="left"
    )
)

mv_training = features_with_mv_split.filter(F.col("mv_pair_level_split") == "train")
mv_validation = features_with_mv_split.filter(F.col("mv_pair_level_split") == "val")
mv_testing = features_with_mv_split.filter(F.col("mv_pair_level_split") == "test")

print("MV proposed split counts")
print("train:", mv_training.count())
print("val:", mv_validation.count())
print("test:", mv_testing.count())

display(
    features_with_mv_split
    .groupBy("mv_pair_level_split")
    .agg(
        F.count("*").alias("num_rows"),
        F.sum(F.col("label")).alias("num_positive"),
        F.mean(F.col("label")).alias("positive_rate"),
        F.countDistinct("login_id").alias("num_employees"),
        F.countDistinct("acct_nbr").alias("num_accounts"),
        F.countDistinct("login_id", "acct_nbr").alias("num_employee_account_pairs")
    )
    .orderBy("mv_pair_level_split")
)

## Notes

- Run cells 1 through 7 to answer the third email question directly.
- Cell 8 prepares `mv_training`, `mv_validation`, and `mv_testing` for a follow-up retraining comparison if the proposed pair-level split is feasible.
- This section should be appended to the end of `lgbm_model.ipynb` after MD's original model training and performance cells.
